[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reinhart-group/generative-copolymer-workshop/blob/main/day1/03_feature_extraction.ipynb)

# Day 1 — Afternoon: CNN Feature Extraction

**Objectives:**
- Understand why simple metrics ($R_g$, aspect ratio) are insufficient for complex aggregates
- Learn how CNNs process spatial data for representation learning
- Compute baseline geometric properties from `.gsd` files
- Use a pre-trained CNN feature extractor to generate high-dimensional feature vectors

In [ ]:
# Install packages not pre-installed on Colab
!pip install -q gsd freud

In [ ]:
import gsd.hoomd
import freud
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision

## 1. Baseline Geometric Properties

TODO: Compute $R_g$, asphericity, etc. from trajectory data

In [ ]:
# Check thermodynamic quantities

fig, ax = plt.subplots(3, 1, figsize=(4, 3),sharex=True)
data = np.loadtxt("block_copolymer.log", skiprows=1)
ax[0].plot(data[:, 0]*0.05, data[:, 3])
ax[0].set_ylabel("potential energy")

ax[1].plot(data[:, 0]*0.05, data[:, 2])
ax[1].set_ylabel("temperature")

ax[2].plot(data[:, 0]*0.05, data[:, 1])
ax[2].set_ylabel("pressure")
ax[2].set_xlabel("time")

plt.show()

## 2. The Limits of Simple Metrics

TODO: Demonstrate where $R_g$ fails to distinguish morphologies

In [ ]:
import gsd, gsd.hoomd

def calc_Rg2(trajectory,N_polymer):
    rg_sq_by_frame = []
    time = []
    for frame in trajectory:
        unwrapped_positions = frame.particles.position + frame.particles.image*frame.configuration.box[0:3]
    # bonds_lengths =  numpy.linalg.norm(unwrapped_positions[frame.bonds.group[:, 0]]- unwrapped_positions[frame.bonds.group[:, 1]],axis=1)
    # all_bonds.append(bonds_lengths)
        rg_sq_this_frame = []

        polymers = np.split(unwrapped_positions, N_polymer)

        for polymer in polymers:

            com = np.mean(polymer, axis=0)
            diff = polymer - com

            rg_sq = np.mean(np.sum(diff**2, axis=1))  # R_g^2

            rg_sq_this_frame.append(rg_sq)

        # Average Rg^2 across all polymers for this frame
        frame_mean = np.mean(rg_sq_this_frame)
        rg_sq_by_frame.append(frame_mean)
        time.append(frame.configuration.step*0.005)
    return time,rg_sq_by_frame

N_polymer = 30

trajectory = gsd.hoomd.open('block_copolymer.gsd',mode='r')
fig, ax = plt.subplots(1, 1, figsize=(4, 3),sharex=True)
time, rg_sq_by_frame = calc_Rg2(trajectory, N_polymer)
ax.plot(time, rg_sq_by_frame)
ax.set_ylabel(" Rg2")

plt.show()

averages = []
sequences= ['AAAAABBBBB','ABABABABA']
# Averaging
for trajectory_file in ['block_copolymer.gsd','alternating_polymer.gsd']:
    trajectory = gsd.hoomd.open(trajectory_file,mode='r')
    time, rg_sq_by_frame = calc_Rg2(trajectory, N_polymer)
    averages.append( np.average(np.asarray(rg_sq_by_frame)[:-500]))

fig, ax = plt.subplots(1, 1, figsize=(4, 3),sharex=True)

ax.scatter([0,1],averages)
plt.show()

## 3. CNN Feature Extraction

TODO: Load pre-trained model and extract feature vectors

In [ ]:
# TODO: Load pre-trained CNN and run feature extraction